In [1]:
import onnx
from onnx import TensorProto, helper

# Create one input (ValueInfoProto)
X1 = helper.make_tensor_value_info("X1", TensorProto.FLOAT, [3, 2])
X2 = helper.make_tensor_value_info("X2", TensorProto.FLOAT, [3, 2])

# Create one output (ValueInfoProto)
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [3, 2])

# Create a node (NodeProto) - This is based on Pad-11
node_def = helper.make_node(
    "Add",  # node name
    ["X1", "X2"],  # inputs
    ["Y"],  # outputs
)

# Create the graph (GraphProto)
graph_def = helper.make_graph(
    [node_def],
    "main_graph",
    [X1, X2],
    [Y],
)


# Set opset version to 18
opset_import = [helper.make_operatorsetid("", 18)]

# Create the model (ModelProto) without using helper.make_model
model_def = helper.make_model(graph_def, producer_name="onnx-example", opset_imports=opset_import)

print(f"The model is:\n{model_def}")

onnx.checker.check_model(model_def)
onnx.save(model_def, "add.onnx")
print("The model is checked!")

The model is:
ir_version: 10
opset_import {
  domain: ""
  version: 18
}
producer_name: "onnx-example"
graph {
  node {
    input: "X1"
    input: "X2"
    output: "Y"
    op_type: "Add"
  }
  name: "main_graph"
  input {
    name: "X1"
    type {
      tensor_type {
        elem_type: 1
        shape {
          dim {
            dim_value: 3
          }
          dim {
            dim_value: 2
          }
        }
      }
    }
  }
  input {
    name: "X2"
    type {
      tensor_type {
        elem_type: 1
        shape {
          dim {
            dim_value: 3
          }
          dim {
            dim_value: 2
          }
        }
      }
    }
  }
  output {
    name: "Y"
    type {
      tensor_type {
        elem_type: 1
        shape {
          dim {
            dim_value: 3
          }
          dim {
            dim_value: 2
          }
        }
      }
    }
  }
}

The model is checked!


In [3]:
import numpy as np
import onnxruntime

np.random.seed(42)

input1 = np.random.rand(3, 2).astype(np.float32)
input2 = np.random.rand(3, 2).astype(np.float32)

# Load the ONNX model
session = onnxruntime.InferenceSession(model_def.SerializeToString())

# Create a dictionary to hold the input data
input_dict = {
    session.get_inputs()[0].name: input1,
    session.get_inputs()[1].name: input2
}

# Run the inference
output = session.run(None, input_dict)

# Output will contain the result of the computation
result = output[0]

# Display the inputs and result
print("Input 1:\n", input1)
print("Input 2:\n", input2)
print("Result:\n", result)


InvalidArgument: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Failed to load model with error: /Users/runner/work/1/s/onnxruntime/core/graph/model.cc:180 onnxruntime::Model::Model(ModelProto &&, const PathString &, const IOnnxRuntimeOpSchemaRegistryList *, const logging::Logger &, const ModelOptions &) Unsupported model IR version: 10, max supported IR version: 9


In [5]:
from xdsl.frontend.onnx import build_module as bm

print(str(bm(model_def.graph)))

builtin.module {
  func.func @main_graph(%0 : tensor<3x2xf32>, %1 : tensor<3x2xf32>) -> tensor<3x2xf32> {
    %2 = onnx.Add(%0, %1) : (tensor<3x2xf32>, tensor<3x2xf32>) -> tensor<3x2xf32>
    func.return %2 : tensor<3x2xf32>
  }
}
